UJI NORMALITAS

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.stats.diagnostic import lilliefors

# ==========================================
# INPUT DATA
# ==========================================
df_tahunan = pd.read_csv('rata-rata-rasio-keuangan.csv')

# ==========================================
# DAFTAR VARIABEL RASIO
# ==========================================
rasio_list = ['ROIC', 'ROA', 'ROE', 'NPM', 'DER', 'TDTC', 'CR', 'FAT']

# ==========================================
# NILAI KRITIS LILLIEFORS D_L(n; α)
# ==========================================

N_TAHUN = 6
ALPHA = 0.05
D_KRITIS_LILLIEFORS = 0.3234

# ==========================================
# PROSES UJI KOLMOGOROV-SMIRNOV DENGAN KOREKSI LILLIEFORS
# ==========================================
hasil_ks = []

for rasio in rasio_list:
    if rasio not in df_tahunan.columns:
        print(f"Peringatan: Kolom '{rasio}' tidak ditemukan di CSV Anda!")
        continue

    data = df_tahunan[rasio].dropna()
    data = data.astype(str).str.replace(',', '.').astype(float)

    if len(data) < 3:
        continue

    # Statistik uji Dn Kolmogorov-Smirnov (parameter mu, sigma diestimasi dari data)
    Dn, _ = lilliefors(data.values, dist='norm')

    keputusan = 'Tolak H0' if Dn >= D_KRITIS_LILLIEFORS else 'Gagal Tolak H0'
    kesimpulan = 'Tidak Normal' if Dn >= D_KRITIS_LILLIEFORS else 'Normal'

    hasil_ks.append({
        'Variabel'              : rasio,
        'Statistik Dn'          : round(Dn, 4),
        f'D^L_({N_TAHUN},{ALPHA}) kritis': D_KRITIS_LILLIEFORS,
        'Keputusan'             : keputusan,
        'Kesimpulan'            : kesimpulan
    })

# ==========================================
# MENAMPILKAN HASIL UJI Normalitas
# ==========================================
df_ks = pd.DataFrame(hasil_ks)
print("=== HASIL UJI NORMALITAS ===")
print("=== (Kolmogorov-Smirnov dengan Koreksi Lilliefors) ===\n")
print(df_ks.to_string(index=False))

=== HASIL UJI NORMALITAS ===
=== (Kolmogorov-Smirnov dengan Koreksi Lilliefors) ===

Variabel  Statistik Dn  D^L_(6,0.05) kritis      Keputusan Kesimpulan
    ROIC        0.2803               0.3234 Gagal Tolak H0     Normal
     ROA        0.2747               0.3234 Gagal Tolak H0     Normal
     ROE        0.2600               0.3234 Gagal Tolak H0     Normal
     NPM        0.2908               0.3234 Gagal Tolak H0     Normal
     DER        0.2156               0.3234 Gagal Tolak H0     Normal
    TDTC        0.1964               0.3234 Gagal Tolak H0     Normal
      CR        0.2017               0.3234 Gagal Tolak H0     Normal
     FAT        0.2300               0.3234 Gagal Tolak H0     Normal


UJI HOMOGENITAS VARIANSI (ANOMV)

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import f as f_dist

# ============================================================
# LOAD DATA
# ============================================================
df_kuartalan = pd.read_csv('hasil-rasio-keuangan.csv')

rasio_list = ['ROIC', 'ROA', 'ROE', 'NPM', 'DER', 'TDTC', 'CR', 'FAT']
tahun_list = [2020, 2021, 2022, 2023, 2024, 2025]

# Fix format koma ke titik
for rasio in rasio_list:
    if rasio in df_kuartalan.columns:
        df_kuartalan[rasio] = df_kuartalan[rasio].astype(str).str.replace(',', '.').astype(float)

# ============================================================
# PARAMETER ANOMV
# ============================================================
I             = 6
n             = 4
N             = I * n        # = 24
vi            = n - 1        # = 3  (df kelompok)
ve            = N - I        # = 18 (df galat)
alpha         = 0.05
alpha_koreksi = alpha / (2 * I)  # = 0.05/12 = 0.00417

UDLv = f_dist.ppf(1 - alpha_koreksi, dfn=vi, dfd=ve)
LDLv = 1 / f_dist.ppf(1 - alpha_koreksi, dfn=ve, dfd=vi)

# ============================================================
# FUNGSI ANOMV
# ============================================================
def anomv(df_kuartalan, rasio, I=6, n=4):
    kelompok = [df_kuartalan[rasio].iloc[i*n:(i+1)*n].values for i in range(I)]
    Si2  = [np.var(k, ddof=1) for k in kelompok]
    SSe  = sum((n - 1) * s for s in Si2)
    MSe  = SSe / (N - I)
    Fi   = [s / MSe for s in Si2]
    return Si2, MSe, Fi

# ============================================================
# OUTPUT: SATU TABEL PER RASIO
# ============================================================
print("=" * 75)
print("            HASIL UJI HOMOGENITAS VARIANSI (ANOMV)")
print(f"  α = {alpha}  |  α/(2I) = {alpha_koreksi:.5f}  |  "
      f"UDLᵥ = {UDLv:.4f}  |  LDLᵥ = {LDLv:.4f}")
print(f"  vᵢ = {vi}  |  vₑ = {ve}")
print("=" * 75)

mse_per_rasio = {}

for rasio in rasio_list:
    Si2, MSe, Fi = anomv(df_kuartalan, rasio)
    mse_per_rasio[rasio] = MSe

    print(f"\n┌─ RASIO: {rasio}   MSe = {MSe:.6f}")
    print(f"  {'Tahun':<8} {'Sᵢ²':>14} {'Fᵢ = Sᵢ²/MSe':>16} {'Keputusan':>15}")
    print(f"  {'─'*55}")
    semua_homogen = True
    for tahun, s, f in zip(tahun_list, Si2, Fi):
        status = "Homogen" if LDLv <= f <= UDLv else "TIDAK Homogen"
        if status != "Homogen":
            semua_homogen = False
        print(f"  {tahun:<8} {s:>14.6f} {f:>16.6f} {status:>15}")
    print(f"  {'─'*55}")
    kesimpulan = "✔ Semua kelompok HOMOGEN" if semua_homogen else "✘ Ada kelompok TIDAK HOMOGEN"
    print(f"  Kesimpulan: {kesimpulan}")

# ============================================================
# RINGKASAN MSE
# ============================================================
print(f"\n{'=' * 75}")
print("  RINGKASAN MSE PER RASIO ")
print(f"{'=' * 75}")
print(f"  {'Rasio':<8} {'MSE':>14}")
print(f"  {'─'*24}")
for rasio, mse in mse_per_rasio.items():
    print(f"  {rasio:<8} {mse:>14.6f}")

            HASIL UJI HOMOGENITAS VARIANSI (ANOMV)
  α = 0.05  |  α/(2I) = 0.00417  |  UDLᵥ = 6.2856  |  LDLᵥ = 0.0206
  vᵢ = 3  |  vₑ = 18

┌─ RASIO: ROIC   MSe = 0.711653
  Tahun               Sᵢ²     Fᵢ = Sᵢ²/MSe       Keputusan
  ───────────────────────────────────────────────────────
  2020           3.318017         4.662411         Homogen
  2021           0.045266         0.063607         Homogen
  2022           0.417220         0.586270         Homogen
  2023           0.021831         0.030677         Homogen
  2024           0.073441         0.103198         Homogen
  2025           0.394139         0.553837         Homogen
  ───────────────────────────────────────────────────────
  Kesimpulan: ✔ Semua kelompok HOMOGEN

┌─ RASIO: ROA   MSe = 0.773339
  Tahun               Sᵢ²     Fᵢ = Sᵢ²/MSe       Keputusan
  ───────────────────────────────────────────────────────
  2020           3.703279         4.788689         Homogen
  2021           0.008034         0.010389   TIDAK 

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =============================
# LOAD DATA
# =============================
df_kuartalan = pd.read_csv('hasil-rasio-keuangan.csv')

rasio_list = ['ROIC', 'ROA', 'ROE', 'NPM', 'DER', 'TDTC', 'CR', 'FAT']
tahun_list = [2020, 2021, 2022, 2023, 2024, 2025]

for rasio in rasio_list:
    if rasio in df_kuartalan.columns:
        df_kuartalan[rasio] = df_kuartalan[rasio].astype(str).str.replace(',', '.').astype(float)

# =============================
# PARAMETER ANOM
# =============================
I = 6
n = 4
N = I * n
h = 2.85

# MSE hasil ANOMV
mse_per_rasio = {
    'ROIC': 0.711648,
    'ROA':  0.773339,
    'ROE':  9.721382,
    'NPM':  91.185331,
    'DER':  128.365079,
    'TDTC': 1.969412,
    'CR':   0.002859,
    'FAT':  0.0000475,
}

# =============================
# FUNGSI ANOM
# =============================
def hitung_anom(df_kuartalan, rasio, MSE, I=6, n=4, N=24, h=2.85):
    """Menghitung grand mean, rata-rata kelompok, Di, dan decision limits ANOM
    (persamaan 2.25-2.33)."""

    grand_mean = df_kuartalan[rasio].sum() / N

    yi_bar = np.array([
        df_kuartalan[rasio].iloc[i * n:(i + 1) * n].mean() for i in range(I)
    ])

    Di = yi_bar - grand_mean

    margin = h * np.sqrt(MSE * (I - 1) / N)
    UDL = grand_mean + margin
    LDL = grand_mean - margin

    return yi_bar, grand_mean, Di, UDL, LDL


# =============================
# DECISION CHART
# =============================
def plot_decision_chart(rasio, tahun_list, yi_bar, GM, UDL, LDL, alpha=0.05, save=True):
    fig, ax = plt.subplots(figsize=(9, 5.5))

    x = np.arange(1, len(tahun_list) + 1)

    ax.axhline(UDL, color='firebrick', linestyle='--', linewidth=1.4, zorder=2,
               label=f'UDL = {UDL:.4f}')
    ax.axhline(GM,  color='seagreen',  linestyle='-',  linewidth=1.4, zorder=2,
               label=f'Grand Mean (CL) = {GM:.4f}')
    ax.axhline(LDL, color='royalblue', linestyle='--', linewidth=1.4, zorder=2,
               label=f'LDL = {LDL:.4f}')

    out_mask = (yi_bar > UDL) | (yi_bar < LDL)
    for xi, yi, out in zip(x, yi_bar, out_mask):
        ax.vlines(xi, GM, yi, color='steelblue', linewidth=1.2, zorder=3)
        if out:
            ax.plot(xi, yi, marker='s', color='firebrick', markersize=9,
                    markeredgecolor='firebrick', zorder=5)
        else:
            ax.plot(xi, yi, marker='o', color='steelblue', markersize=8,
                    markerfacecolor='white', markeredgewidth=1.6, zorder=5)
        offset = 10 if yi >= GM else -14
        va = 'bottom' if yi >= GM else 'top'
        ax.annotate(f'{yi:.4f}', xy=(xi, yi), xytext=(0, offset),
                    textcoords='offset points', ha='center', va=va,
                    fontsize=8.5, fontweight='bold' if out else 'normal',
                    color='firebrick' if out else 'black')

    ax.set_title(f'Decision Chart ANOM — {rasio}', fontsize=13, fontweight='bold', pad=28)
    ax.text(0.5, 1.03, f'α = {alpha}', transform=ax.transAxes,
            ha='center', fontsize=10, color='dimgray')
    ax.set_xlabel('Tahun', fontsize=11)
    ax.set_ylabel(f'Rata-rata {rasio}', fontsize=11)
    ax.set_xticks(x)
    ax.set_xticklabels(tahun_list)
    ax.grid(True, alpha=0.25)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=3,
              fontsize=9, frameon=False)

    y_all = np.concatenate([yi_bar, [UDL, LDL]])
    pad = (y_all.max() - y_all.min()) * 0.15
    ax.set_ylim(y_all.min() - pad, y_all.max() + pad)
    plt.tight_layout()
    if save:
        plt.savefig(f'decision_chart_{rasio}.png', dpi=150, bbox_inches='tight')
    return fig


# =============================
# OUTPUT TABEL + DECISION CHART PER RASIO
# =============================
print("=" * 70)
print("           HASIL ANALISIS ANOM KINERJA KEUANGAN")
print(f"  h = {h}  |  α = 0.05  |  I = {I}  |  n = {n}  |  N = {N}  |  ν = {N-I}")
print("  Rumus UDL/LDL: ȳ.. ± h · √(MSE · (I-1)/N)  [Persamaan 2.32-2.33]")
print("=" * 70)

ringkasan = []
for rasio in rasio_list:
    MSE = mse_per_rasio[rasio]
    yi_bar, GM, Di, UDL, LDL = hitung_anom(df_kuartalan, rasio, MSE)

    print(f"\n┌─ RASIO: {rasio}")
    print(f"  Grand Mean (ȳ..) = {GM:.4f}  |  UDL = {UDL:.4f}  |  LDL = {LDL:.4f}")
    print(f"  {'Tahun':<8} {'ȳᵢ':>10} {'Dᵢ = ȳᵢ - ȳ..':>16} {'Keputusan':>18}")
    print(f"  {'─'*54}")

    ada_anomali = False
    for i, tahun in enumerate(tahun_list):
        if yi_bar[i] > UDL:
            status = "Signifikan tinggi"
            ada_anomali = True
        elif yi_bar[i] < LDL:
            status = "Signifikan rendah"
            ada_anomali = True
        else:
            status = "Dalam batas"
        print(f"  {tahun:<8} {yi_bar[i]:>10.4f} {Di[i]:>16.4f} {status:>18}")

    print(f"  {'─'*54}")
    kesimpulan = "Ada kelompok yang menyimpang signifikan" if ada_anomali \
                 else "Tidak ada kelompok yang menyimpang signifikan"
    print(f"  Kesimpulan: {kesimpulan}")
    ringkasan.append((rasio, GM, UDL, LDL, ada_anomali))

    plot_decision_chart(rasio, tahun_list, yi_bar, GM, UDL, LDL)
    plt.close()
    print(f"  → Decision chart disimpan: decision_chart_{rasio}.png")

# =============================
# RINGKASAN
# =============================
print(f"\n{'=' * 70}")
print("  RINGKASAN HASIL ANOM — SEMUA RASIO")
print(f"{'=' * 70}")
print(f"  {'Rasio':<8} {'Grand Mean':>12} {'UDL':>10} {'LDL':>10} {'Kesimpulan':>35}")
print(f"  {'─'*72}")
for rasio, GM, UDL, LDL, anomali in ringkasan:
    ket = "Terdapat kelompok signifikan" if anomali else "Tidak ada kelompok signifikan"
    print(f"  {rasio:<8} {GM:>12.4f} {UDL:>10.4f} {LDL:>10.4f} {ket:>35}")

           HASIL ANALISIS ANOM KINERJA KEUANGAN
  h = 2.85  |  α = 0.05  |  I = 6  |  n = 4  |  N = 24  |  ν = 18
  Rumus UDL/LDL: ȳ.. ± h · √(MSE · (I-1)/N)  [Persamaan 2.32-2.33]

┌─ RASIO: ROIC
  Grand Mean (ȳ..) = 0.9366  |  UDL = 2.0340  |  LDL = -0.1608
  Tahun            ȳᵢ    Dᵢ = ȳᵢ - ȳ..          Keputusan
  ──────────────────────────────────────────────────────
  2020         0.1591          -0.7775        Dalam batas
  2021         1.1783           0.2417        Dalam batas
  2022         0.8991          -0.0375        Dalam batas
  2023         1.2089           0.2723        Dalam batas
  2024         1.2933           0.3567        Dalam batas
  2025         0.8810          -0.0556        Dalam batas
  ──────────────────────────────────────────────────────
  Kesimpulan: Tidak ada kelompok yang menyimpang signifikan
  → Decision chart disimpan: decision_chart_ROIC.png

┌─ RASIO: ROA
  Grand Mean (ȳ..) = 0.2757  |  UDL = 1.4197  |  LDL = -0.8682
  Tahun            ȳᵢ    Dᵢ =